![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [1]:
from apple_mujoco_setup import *

Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
Detected macOS: arm64
Using MUJOCO_GL=glfw for macOS
Forcing JAX to run on CPU backend
Mujoco installation and rendering backend OK
JAX: 0.6.2
MuJoCo: 3.3.6
Brax: 0.13.0
Flax: 0.10.7

Checking media packages...
✓ ffmpeg available
✓ mediapy available
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
JAX device count: 8


# Manipulation

MuJoCo Playground contains several manipulation environments (all listed below after running the command).

In [2]:
registry.manipulation.ALL_ENVS

('AlohaHandOver',
 'AlohaSinglePegInsertion',
 'PandaPickCube',
 'UR10PickCube',
 'PandaPickCubeOrientation',
 'PandaPickCubeCartesian',
 'PandaOpenCabinet',
 'PandaRobotiqPushCube',
 'LeapCubeReorient',
 'LeapCubeRotateZAxis')

# UR10e with Hand E - Run Diagnostics to xml

Let's start off with the simplest environment, simply picking up a cube with the Franka Emika Panda.

In [3]:
env_name = 'UR10PickCube'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)
print("Loaded:", env)
print("Action size:", env.action_size)
print("Gripper site ID:", env._gripper_site)

m = env.mj_model

print("\nBodies:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(m.nbody)])
print("Joints:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(m.njnt)])
print("Actuators:", env.mj_model.nu, [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(m.nu)])
print("Sites:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_SITE, i) for i in range(m.nsite)])
print(env.action_size)
print([env.mj_model.joint(i).name for i in range(env.mj_model.njnt)])

/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "coupler" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(


✓ Using keyframe: 'task_home'
  Initial qpos size: 15
  Robot joints: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.      0.    ]


KeyError: "Invalid name 'left_finger_touch_floor_found'. Valid names: ['box_position', 'gripper_floor_contact', 'gripper_force', 'left_finger_touch', 'right_finger_touch', 'tcp_position']"

In [ ]:
print("Actuators:", env.mj_model.nu)
print([env.mj_model.actuator(i).name for i in range(env.mj_model.nu)])

: 

## Rollout before training


In [ ]:
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml'
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/test_grvity.xml'
# '/../mujoco_playground/external_deps/mujoco_menagerie/franka_emika_panda/scene.xml'
# '../../mujoco_playground/_src/manipulation/my_ur10/universal_robots_ur10e/ur10e.xml'

: 

## Gravity tests and rollouts

There needs to be the torque model to see the Robot fall. If it is position controll nothing works

In [ ]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene_torque.xml')
data = mujoco.MjData(model)

# Load gravity test keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'gravity_test')
mujoco.mj_resetDataKeyframe(model, data, key_id)

print("Launching viewer with gravity test...")
print("Robot will fall under gravity (ctrl=0 = zero torque)")
print("Press Space to start/stop simulation")

mujoco.viewer.launch(model, data)

: 

## Test Movements of Actuators

In [ ]:
model = mujoco.MjModel.from_xml_path("../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene_position.xml")
data = mujoco.MjData(model)

mujoco.viewer.launch(model, data)

: 

## Test floor collision and gripper collision


In [ ]:
m = env.mj_model

def enum_name(enum_cls, value_array):
    # value_array may be a 0-d numpy array; convert to a Python int first
    v = int(np.asarray(value_array).item())
    return enum_cls(v).name

rows = []
for i in range(m.nsensor):
    s = m.sensor(i)
    s_name = s.name
    s_type = enum_name(mujoco.mjtSensor, s.type)
    objtype_int = int(np.asarray(s.objtype).item())
    objtype_name = mujoco.mjtObj(objtype_int).name
    objid_int = int(np.asarray(s.objid).item())
    target_name = mujoco.mj_id2name(m, objtype_int, objid_int) if objid_int >= 0 else None
    dim = list(m.sensor_dim[i:i+1]) if hasattr(m, "sensor_dim") else list(s.dim)

    rows.append((i, s_name, s_type, objtype_name, target_name, dim))

print(f"{'id':>3} | {'name':<28} | {'type':<15} | {'objtype':<12} | {'target':<28} | dim")
print("-"*100)
for r in rows:
    print(f"{r[0]:>3} | {r[1]:<28} | {r[2]:<15} | {r[3]:<12} | {str(r[4]):<28} | {r[5]}")



: 

In [ ]:
m = env.mj_model  # compiled MjModel

# If you don't have a state yet in this kernel:
jit_reset = jax.jit(env.reset)
state = jit_reset(jax.random.PRNGKey(0))

# Geom ids (do this once)
g_fingerL = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, "left_finger-collision")
g_floor   = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, "floor")

def contact_normal_force_finger_floor(state) -> float:
    """Return total normal force [N] between left_finger-collision and floor."""

    # MJX -> numpy
    qpos = np.array(state.data.qpos)   # JAX array -> numpy
    qvel = np.array(state.data.qvel)

    # Build a temporary MuJoCo data snapshot and evaluate contacts
    d = mujoco.MjData(m)
    d.qpos[:] = qpos
    d.qvel[:] = qvel
    mujoco.mj_forward(m, d)

    total_fn = 0.0
    # iterate through all contacts
    for i in range(d.ncon):
        con = d.contact[i]
        if {con.geom1, con.geom2} == {g_fingerL, g_floor}:
            # Resolved contact force in the contact frame:
            # result[0:3] = force components [normal, tangent1, tangent2]
            # result[3:6] = torque components
            cf = np.zeros(6, dtype=float)
            mujoco.mj_contactForce(m, d, i, cf)
            fn = max(0.0, cf[0])  # normal component only
            total_fn += fn
    return total_fn

# Example usage:
fn_now = contact_normal_force_finger_floor(state)
print("Left finger ↔ floor normal force [N]:", fn_now)



: 

## Train Policy

Let's train the pick cube policy and visualize rollouts. The policy takes roughly 3 minutes to train on an RTX 4090.

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params

: 

In [ ]:
from copy import deepcopy

# Make a copy so you don't overwrite the original defaults
fast_ppo_params = deepcopy(ppo_params)

# --- Speedup adjustments ---
fast_ppo_params["num_timesteps"] = 10_000_000 
# fast_ppo_params["learning_rate"] = 0.002        
# fast_ppo_params["episode_length"] = 200        
# fast_ppo_params["unroll_length"] = 10         

# (Optional) reduce parallel envs to lower compute requirements
# fast_ppo_params["num_envs"] = 2048              # from 8192 or 4096 or 2048
# print("Adjusted num_envs:", fast_ppo_params["num_envs"])

# (Optional) tweak batch sizes accordingly !! Needs to be 
# fast_ppo_params["batch_size"] = 256             
# fast_ppo_params["num_minibatches"] = 16         

ppo_params = fast_ppo_params
ppo_params

: 

### PPO

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1
)

: 

In [ ]:
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
if y_data:
    import numpy as np
    best_idx = int(np.argmax(y_data))

: 

### Metrics of Training
Rewards over 1000 yield decent results
With default training time reward is 1347 after 20'152'320

In [ ]:
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")
print(f"Highest Reward: {y_data[best_idx]:.3f} ± {y_dataerr[best_idx]:.3f} at step {x_data[best_idx]}")

: 

## Visualize Rollouts

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

: 

In [ ]:
rng = jax.random.PRNGKey(42)
rollout = []
n_episodes = 1

for _ in range(n_episodes):
  state = jit_reset(rng)
  rollout.append(state)
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

: 

While the above policy is very simple, the work was extended using the Madrona batch renderer, and policies were transferred on a real robot. We encourage folks to check out the Madrona-MJX tutorial notebooks ([part 1](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_1.ipynb) and [part 2](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_2.ipynb))!